# Database Hub demo: on-demand activity control

Run this notebook to **start the demo's activity when you want it** and **watch it live**. It calls the demo's Azure Automation webhook (ADR 0020), whose runbook starts the targets that are stopped and runs the bounded scenarios you choose; then it shows each target's waits, blocking, CPU, and IO from the same telemetry as Database Hub **Performance**.

**How it runs only when you run it.** Nothing is scheduled: the runbook starts only when this notebook calls its webhook. It checks every request against its allow-lists, refuses a run while another one is active, caps each scenario at its maximum and before the VMs' nightly auto-shutdown, and creates a one-time **auto-stop** 10 minutes after the run's end, before it starts anything.

**Stop layers.** Each scenario's maximum duration; the run's auto-stop, which deallocates both VMs and stops T4; **Stop now** in section 4 (or `./scripts/stop-demo.ps1`); the VMs' nightly auto-shutdown; T3's auto-pause; and T4's daily safety stop. None of them uses Run Command.

**Safety.** The webhook's URL is a secret: this notebook reads it from Key Vault for one request and never prints it. **Run all** sends nothing: the two **Resource-changing** cells print a dry run unless `CONFIRM = True`, and Stop now and Reset also need their own switch; with one of those set, the run cell sends nothing. The live view only reads.

> **Preview disclosure:** Database Hub and its telemetry endpoint are previews. Using a notebook `kusto` token against the endpoint isn't documented, so when it refuses the token the live view prints its KQL for the Azure Data Explorer web UI. Telemetry arrives 3 to 15 minutes after the activity.

**Contents:** 1 Parameters · 2 Helpers · 3 Run · 4 Stop now and Reset · 5 Live view · 6 Links

## 1. Parameters

`setup-activity-control.ps1` filled the environment values when it imported this notebook. Choose the activity: one scenario per target, `baseline` or a scenario from the catalog (`CATALOG_JSON`), and how long it runs.

In [ ]:
# Parameters cell (tagged "parameters"). setup-activity-control.ps1 fills the environment values when it imports
# this notebook. Never put a credential here: the webhook URL stays in Key Vault.

# Environment (filled at import).
KEY_VAULT_URL = ""                               # https://kv-<prefix>-ctl-<suffix>.vault.azure.net/
WEBHOOK_SECRET_NAME = "dbhub-activity-webhook"   # the secret's name, never its value
RESOURCE_GROUP = "rg-dbhub-demo"
T1_VM_NAME = "sqlvm01"
T2_MACHINE_NAME = "arcsql01"
T3_SERVER_PREFIX = "azuresql01-"
DEMO_DATABASE = "ContosoCare"
CATALOG_JSON = ""                                # the runbook's allow-lists: targets, scenarios, and bounds

# The activity: one entry per target, "baseline" or a scenario from the catalog. "intensity" (1-10) and, for an
# incident, "concurrency" (1 to the scenario's maximum) are optional. A pipeline or the Fabric job API passes
# ACTIVITIES and RESET_TARGETS as JSON text, because a notebook parameter is a string, a number, or a bool.
ACTIVITIES = [
    {"target": "T1", "scenario": "blocking-chain"},
    {"target": "T2", "scenario": "tempdb-contention"},
    {"target": "T3", "scenario": "baseline"},
]
DURATION_MINUTES = 20       # 5-120; each incident also stops at its own maximum and before the VMs' auto-shutdown
RESET_TARGETS = []          # Reset only: the targets to reset, for example ["T1"] (empty: every target whose VM runs)

# Safety switches: nothing is sent unless CONFIRM is True; Stop now and Reset also need their own switch.
CONFIRM = False
STOP_NOW = False
RESET_NOW = False

# Live view.
WATCH_MINUTES = 20          # how long the live view keeps refreshing (0: one look)
REFRESH_SECONDS = 60        # 30-300
LOOKBACK_MINUTES = 15       # the window each refresh shows

## 2. Helpers

Run the next two cells first. They change nothing:

- **Requests:** `build_request` checks the activity against the catalog the runbook allows, the same rules the runbook applies again, and builds the webhook's JSON body: `{"action": "run", "activities": [...], "durationMinutes": n}`, `{"action": "stop"}`, or `{"action": "reset", "targets": [...]}`. `send_request` reads the webhook URL with `notebookutils.credentials.getSecret` for that one request, posts the body with no credential header, and prints only the HTTP status and the Automation job's ID.
- **Telemetry:** `notebookutils.credentials.getToken("kusto")` for the documented telemetry endpoint, and the same KQL as the live-activity notebook and the Real-Time Dashboard (`scripts/modules/DbHubDemo/Kql`).

In [ ]:
# Helpers, part 1: requests. No network calls; the webhook is called only from sections 3 and 4.
import json
import re

ACTIONS = ("run", "stop", "reset")
MAX_BODY_BYTES = 4096
_NAME = re.compile(r"^[A-Za-z0-9._@/-]{1,64}$")
_REDACTIONS = (
    (re.compile(r"eyJ[A-Za-z0-9_-]{5,}\.[A-Za-z0-9_-]{5,}\.[A-Za-z0-9_-]*"), "[REDACTED:jwt]"),
    (re.compile(r"(?i)(bearer\s+)[A-Za-z0-9._~+/=-]{8,}"), r"\1[REDACTED]"),
    (re.compile(r"https://[A-Za-z0-9.-]+\.azure-automation\.net/\S*"), "[REDACTED:webhook]"),
    (re.compile(r"(?i)(webhooks\?token=)[^&\s\"']+"), r"\1[REDACTED]"),
)


def redact(value):
    """Removes token-like strings and webhook URLs from text before it's printed."""
    text = "" if value is None else str(value)
    for pattern, replacement in _REDACTIONS:
        text = pattern.sub(replacement, text)
    return text


def load_catalog(text):
    """The runbook's allow-lists from CATALOG_JSON: targets, scenarios (maxMinutes, maxConcurrency, targets), and bounds."""
    if not text:
        raise ValueError("CATALOG_JSON is empty: import this notebook with setup-activity-control.ps1.")
    catalog = json.loads(text)
    for key in ("targets", "scenarios", "minDurationMinutes", "maxDurationMinutes"):
        if key not in catalog:
            raise ValueError("CATALOG_JSON has no %s." % key)
    return catalog


def as_list(value, name):
    """A list parameter, as a Python list or as JSON text: a run through the Fabric job API or a pipeline gets each
    notebook parameter as a string, a number, or a bool."""
    if value is None:
        return []
    if isinstance(value, str):
        if not value.strip():
            return []
        try:
            value = json.loads(value)
        except ValueError:
            raise ValueError("%s must be a list, or a JSON array as text." % name) from None
    if not isinstance(value, (list, tuple)):
        raise ValueError("%s must be a list, or a JSON array as text." % name)
    return list(value)


def _whole(value, low, high, name):
    if isinstance(value, bool) or not isinstance(value, int) or not low <= value <= high:
        raise ValueError("%s must be a whole number from %d to %d." % (name, low, high))
    return value


def build_request(action, activities=None, duration_minutes=None, reset_targets=None, catalog=None, requested_by="notebook"):
    """Validates a request with the runbook's rules and returns the webhook body (a dict). No network calls."""
    if action not in ACTIONS:
        raise ValueError("action must be run, stop, or reset.")
    if not _NAME.match(str(requested_by)):
        raise ValueError("requested_by must be 1-64 letters, digits, or . _ @ / -.")
    body = {"action": action, "requestedBy": str(requested_by)}
    if action == "stop":
        return body
    targets = list(catalog["targets"])
    if action == "reset":
        chosen = as_list(reset_targets, "RESET_TARGETS")
        if len(set(chosen)) != len(chosen) or any(target not in targets for target in chosen):
            raise ValueError("RESET_TARGETS must be distinct values from: %s." % ", ".join(targets))
        if chosen:
            body["targets"] = chosen
        return body
    body["durationMinutes"] = _whole(duration_minutes, int(catalog["minDurationMinutes"]), int(catalog["maxDurationMinutes"]), "DURATION_MINUTES")
    items = as_list(activities, "ACTIVITIES")
    if not 1 <= len(items) <= len(targets):
        raise ValueError("ACTIVITIES needs 1 to %d entries, one per target." % len(targets))
    seen = set()
    cleaned = []
    for item in items:
        if not isinstance(item, dict):
            raise ValueError("Each activity is an object with a target and a scenario.")
        unknown = set(item) - {"target", "scenario", "intensity", "concurrency"}
        if unknown:
            raise ValueError("An activity can't have: %s." % ", ".join(sorted(unknown)))
        target, scenario = item.get("target"), item.get("scenario")
        if target not in targets:
            raise ValueError("Each target must be one of: %s." % ", ".join(targets))
        if target in seen:
            raise ValueError("%s has more than one activity; a target runs one at a time." % target)
        seen.add(target)
        entry = {"target": target, "scenario": scenario}
        limit = 0
        if scenario != "baseline":
            definition = catalog["scenarios"].get(scenario)
            if definition is None:
                raise ValueError("'%s' isn't in the catalog: use baseline or one of %s." % (scenario, ", ".join(sorted(catalog["scenarios"]))))
            if target not in definition["targets"]:
                raise ValueError("%s doesn't run on %s." % (scenario, target))
            limit = int(definition["maxConcurrency"])
        if "intensity" in item:
            entry["intensity"] = _whole(item["intensity"], 1, 10, "intensity")
        if "concurrency" in item:
            if not limit:
                raise ValueError("concurrency applies to incidents only.")
            entry["concurrency"] = _whole(item["concurrency"], 1, limit, "concurrency")
        cleaned.append(entry)
    body["activities"] = cleaned
    return body


def describe_plan(body, catalog):
    """One line per activity: how long it runs, with the scenario's own maximum (the runbook also stops it before the
    VMs' auto-shutdown)."""
    lines = []
    for item in body.get("activities", []):
        cap = int(catalog["maxDurationMinutes"]) if item["scenario"] == "baseline" else int(catalog["scenarios"][item["scenario"]]["maxMinutes"])
        lines.append("%s: %s for %d minutes" % (item["target"], item["scenario"], min(body["durationMinutes"], cap)))
    return lines

In [ ]:
# Helpers, part 2: the webhook call and telemetry queries. A token or the webhook URL is used for one request and
# never printed, logged, returned, or saved.
import datetime as dt
import time

import pandas as pd
import requests

TELEMETRY_QUERY_URL = "https://adx.centralus.arcdataservices.com/kusto/v1/rest/query"
TELEMETRY_CONNECTION_URI = "https://adx.centralus.arcdataservices.com/kusto/"
TELEMETRY_DATABASE = "ArcSqlTelemetry"
ADX_WEB_UI = "https://dataexplorer.azure.com/"
DATABASE_HUB_URL = "https://powerbi.com/workloads/fdh/databaseHub"
WEBHOOK_HTTP_HINTS = {
    400: "HTTP 400: the webhook has expired or is disabled (setup-activity-control.ps1 -RotateWebhook replaces it).",
    404: "HTTP 404: the webhook, its runbook, or the Automation account wasn't found; run setup-activity-control.ps1.",
    500: "HTTP 500: Azure Automation couldn't take the request; try again in a minute.",
}
TELEMETRY_HTTP_HINTS = {
    400: "HTTP 400: the KQL has an error, or a table or column changed during the preview.",
    401: "HTTP 401: the endpoint didn't accept the notebook's kusto token, a path that isn't documented.",
    403: "HTTP 403: you need Reader on the subscription, and Microsoft.Sql must be registered on it.",
}


def _notebookutils():
    try:
        return notebookutils  # noqa: F821 - Fabric provides notebookutils in every notebook session.
    except NameError:
        try:
            import notebookutils as module
            return module
        except ImportError:
            return None


def send_request(body):
    """Posts one request to the webhook. Prints the HTTP status and the job ID only; returns the job IDs or None."""
    payload = json.dumps(body, separators=(",", ":"))
    if len(payload.encode("utf-8")) > MAX_BODY_BYTES:
        raise ValueError("The request is larger than %d bytes." % MAX_BODY_BYTES)
    if not KEY_VAULT_URL:
        print("Manual action required: KEY_VAULT_URL is empty. Import this notebook with setup-activity-control.ps1.")
        return None
    utils = _notebookutils()
    if utils is None:
        print("Run this notebook in Microsoft Fabric: notebookutils isn't available here.")
        return None
    url = None
    try:
        url = utils.credentials.getSecret(KEY_VAULT_URL, WEBHOOK_SECRET_NAME)
    except Exception as error:  # the message is redacted; the URL is never in it, but a vault name could be
        print("The webhook URL couldn't be read from Key Vault (%s). You need Key Vault Secrets User on the secret "
              "(setup-activity-control.ps1 -GrantRole), and a private vault is reached only from the DBHub Control "
              "workspace." % type(error).__name__)
        return None
    try:
        response = requests.post(url, data=payload, headers={"Content-Type": "application/json"}, timeout=60)
    except requests.RequestException as error:
        print("The webhook couldn't be reached (%s); the error text isn't printed, because it can hold the URL." % type(error).__name__)
        return None
    finally:
        url = None
    if response.status_code == 202:
        try:
            jobs = list(response.json().get("JobIds", []))
        except ValueError:
            jobs = []
        print("Accepted (HTTP 202): Automation job %s. The runbook checks the request again; its output is in the "
              "Automation account's job." % (", ".join(jobs) or "(no ID returned)"))
        return jobs
    print(WEBHOOK_HTTP_HINTS.get(response.status_code, "HTTP %d from the webhook." % response.status_code))
    return None


def get_token(audience):
    """A token for a documented notebookutils audience key. Never print, log, or save the result."""
    if audience != "kusto":
        raise ValueError("This notebook asks only for the kusto audience.")
    utils = _notebookutils()
    if utils is None:
        raise RuntimeError("notebookutils isn't available. Run this notebook in Microsoft Fabric.")
    return utils.credentials.getToken(audience)


def kql_string(value):
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"


def kql_datetime(value):
    return "datetime(" + value.astimezone(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ") + ")"


def kql_prelude(start, end, bin_minutes=1):
    """Time window, bin, names, and a function that maps telemetry rows to T1, T2, or T3 (Kql/Live.Prelude.kql)."""
    return "\n".join([
        "let _start = %s;" % kql_datetime(start),
        "let _end = %s;" % kql_datetime(end),
        "let _bin = %dm;" % int(bin_minutes),
        "let _rg = %s;" % kql_string(RESOURCE_GROUP),
        "let _db = %s;" % kql_string(DEMO_DATABASE),
        "let _t1 = %s;" % kql_string(T1_VM_NAME),
        "let _t2 = %s;" % kql_string(T2_MACHINE_NAME),
        "let _t3prefix = %s;" % kql_string(T3_SERVER_PREFIX),
        "let _t3suffix = %s;" % kql_string("/" + DEMO_DATABASE),
        "let _target = (resourceType: string, resourceName: string) {",
        "    case(resourceType == 'virtualMachines' and resourceName =~ _t1, 'T1',",
        "         resourceType == 'machines' and resourceName =~ _t2, 'T2',",
        "         resourceType == 'servers/databases' and resourceName startswith _t3prefix and resourceName endswith _t3suffix, 'T3',",
        "         '')",
        "};",
    ])


def _to_utc(series):
    try:
        return pd.to_datetime(series, utc=True, format="ISO8601")
    except (TypeError, ValueError):
        return pd.to_datetime(series, utc=True, errors="coerce")


def kusto_frame(body):
    """Turns a Kusto v1 response (Tables[0].Columns and Rows) into a DataFrame."""
    tables = body.get("Tables") or []
    if not tables:
        return pd.DataFrame()
    first = tables[0]
    columns = first.get("Columns", [])
    frame = pd.DataFrame(first.get("Rows", []), columns=[column.get("ColumnName") for column in columns])
    for column in columns:
        if str(column.get("ColumnType") or column.get("DataType")).lower() in ("datetime", "system.datetime"):
            frame[column["ColumnName"]] = _to_utc(frame[column["ColumnName"]])
    return frame


def post_kql(kql):
    """Sends read-only KQL to the telemetry endpoint. Returns (status, frame, message)."""
    try:
        token = get_token("kusto")
    except Exception as error:
        return 0, None, "notebookutils didn't return a kusto token: " + redact(error)
    try:
        response = requests.post(TELEMETRY_QUERY_URL, json={"db": TELEMETRY_DATABASE, "csl": str(kql)},
                                 headers={"Authorization": "Bearer " + token, "Content-Type": "application/json"}, timeout=180)
    except requests.RequestException as error:
        return 0, None, "The telemetry endpoint couldn't be reached: " + redact(error)
    finally:
        token = None
    if response.status_code == 200:
        try:
            return 200, kusto_frame(response.json()), ""
        except ValueError:
            return 0, None, "The telemetry endpoint returned something other than Kusto v1 JSON."
    return response.status_code, None, TELEMETRY_HTTP_HINTS.get(response.status_code, "HTTP %d." % response.status_code)


# The same KQL as scripts/modules/DbHubDemo/Kql/Live.*.kql, without comments; tests/unit/Fabric.ControlNotebook.Tests.ps1
# fails if they drift.
CPU_SUMMARY_BODY = """
SqlServerCPUUtilization
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| extend CpuPercent = iff(ResourceType =~ 'servers/databases', AvgCPUPercent, coalesce(todouble(SQLProcessCPUPercent), AvgCPUPercent))
| summarize AvgCPU = round(avg(CpuPercent), 1), P95CPU = round(percentile(CpuPercent, 95), 1),
            MaxCPU = round(max(CpuPercent), 1), PctTimeAbove80 = round(100.0 * countif(CpuPercent > 80.0) / count(), 1)
        by Target, ResourceType, ResourceName, SqlServerInstanceName
| order by Target asc"""
WAITS_BODY = """
SqlServerWaitStats
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| where WaitCategory !in ('Idle', 'Unknown')
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize (FirstT, FirstW, FirstC) = arg_min(SampleTimeUTC, WaitTimeMs, WaitingTasksCount),
            (LastT, LastW, LastC) = arg_max(SampleTimeUTC, WaitTimeMs, WaitingTasksCount)
        by Target, ResourceID, SqlServerInstanceName, WaitCategory, WaitType, Bucket = bin(SampleTimeUTC, _bin)
| extend T = pack_array(FirstT, LastT), W = pack_array(FirstW, LastW), C = pack_array(FirstC, LastC)
| mv-expand T to typeof(datetime), W to typeof(long), C to typeof(long)
| distinct Target, ResourceID, SqlServerInstanceName, WaitCategory, WaitType, Bucket, T, W, C
| sort by ResourceID asc, SqlServerInstanceName asc, WaitType asc, T asc
| extend Keep = ResourceID == prev(ResourceID) and SqlServerInstanceName == prev(SqlServerInstanceName)
                and WaitType == prev(WaitType) and W >= prev(W) and C >= prev(C)
| extend WaitMs = iff(Keep, W - prev(W), long(0)), Tasks = iff(Keep, C - prev(C), long(0))
| summarize WaitMs = sum(WaitMs), Tasks = sum(Tasks) by Target, WaitCategory, PageIoLatch = WaitType startswith 'PAGEIOLATCH', Bucket
| where WaitMs > 0
| order by Bucket asc
| take 50000"""
GAUGES_BODY = """
SqlServerPerformanceCountersCommon
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| where CounterType == 65792
| where CounterName in ('Processes blocked', 'Active Transactions', 'User Connections')
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize MaxValue = max(CounterValue), AvgValue = avg(CounterValue)
        by Target, ResourceName, SqlServerInstanceName, CounterName, Bucket = bin(SampleTimeUTC, _bin)
| order by Bucket asc
| take 50000"""


def live_snapshot(lookback_minutes):
    """The last lookback_minutes per target: CPU, the top wait categories, and blocking. Returns (frames, message)."""
    end = dt.datetime.now(dt.timezone.utc)
    start = end - dt.timedelta(minutes=int(lookback_minutes))
    prelude = kql_prelude(start, end, 1)
    frames = {}
    for key, body in (("cpu", CPU_SUMMARY_BODY), ("waits", WAITS_BODY), ("gauges", GAUGES_BODY)):
        status, frame, message = post_kql(prelude + body)
        if status != 200:
            return None, message + " Paste this KQL into the Azure Data Explorer web UI (%s, connection %s, database %s):\n%s" % (ADX_WEB_UI, TELEMETRY_CONNECTION_URI, TELEMETRY_DATABASE, prelude + body)
        frames[key] = frame
    return frames, ""


def show_snapshot(frames):
    """Prints one compact view per refresh: CPU, the top wait categories, and blocked processes per target."""
    cpu, waits, gauges = frames["cpu"], frames["waits"], frames["gauges"]
    if not cpu.empty:
        print(cpu[["Target", "AvgCPU", "MaxCPU", "PctTimeAbove80"]].to_string(index=False))
    if not waits.empty:
        top = (waits.groupby(["Target", "WaitCategory"], as_index=False)["WaitMs"].sum()
               .assign(WaitSeconds=lambda f: (f["WaitMs"] / 1000.0).round(1))
               .sort_values(["Target", "WaitSeconds"], ascending=[True, False]).groupby("Target").head(5))
        print(top[["Target", "WaitCategory", "WaitSeconds"]].to_string(index=False))
    if not gauges.empty:
        blocked = gauges[gauges["CounterName"] == "Processes blocked"].groupby("Target", as_index=False)["MaxValue"].max()
        print(blocked.rename(columns={"MaxValue": "MaxBlockedProcesses"}).to_string(index=False))
    if cpu.empty and waits.empty and gauges.empty:
        print("No telemetry in the window yet: it arrives 3 to 15 minutes after the activity.")

## 3. Run

**Resource-changing.** With `CONFIRM = True`, the next cell posts the activity to the webhook. The runbook refuses it while another run is active; otherwise it creates the run's auto-stop, starts the stopped VMs (and T4), and sets each target's activity profile through one Run Command per VM, which it doesn't wait for. With `STOP_NOW` or `RESET_NOW` set, this cell sends nothing, so **Run all** (or a run through the Fabric job API) with one of them sends only the stop or the reset. The runbook's output is in the Automation account's job; this notebook can't read it, because it has no Azure Resource Manager token.

In [ ]:
# Resource-changing: starts the activity only when CONFIRM is True; otherwise a dry run.
CATALOG = load_catalog(CATALOG_JSON)
REQUEST = build_request("run", ACTIVITIES, DURATION_MINUTES, catalog=CATALOG)
for line in describe_plan(REQUEST, CATALOG):
    print(line)
if CONFIRM and (STOP_NOW or RESET_NOW):
    print("Stop now or Reset is set, so this cell sends no run request; section 4 sends that one.")
elif CONFIRM:
    JOBS = send_request(REQUEST)
else:
    print("Dry run: nothing was sent. Set CONFIRM = True and run this cell again to start it.")

## 4. Stop now and Reset

**Resource-changing.** **Stop now** removes the run's auto-stop, deallocates both VMs, and stops T4 through Azure Resource Manager only; it needs `STOP_NOW = True` and `CONFIRM = True`. **Reset** runs the runner's reset on the targets whose VM runs (`RESET_TARGETS`, or all of them) and needs `RESET_NOW = True` and `CONFIRM = True`. `./scripts/stop-demo.ps1` stops the demo from your machine too.

In [ ]:
# Resource-changing: Stop now and Reset, each only with its switch and CONFIRM.
CATALOG = load_catalog(CATALOG_JSON)
if STOP_NOW:
    if CONFIRM:
        send_request(build_request("stop"))
    else:
        print("Dry run: Stop now needs CONFIRM = True too.")
if RESET_NOW:
    if CONFIRM:
        send_request(build_request("reset", reset_targets=RESET_TARGETS, catalog=CATALOG))
    else:
        print("Dry run: Reset needs CONFIRM = True too.")
if not STOP_NOW and not RESET_NOW:
    print("Nothing to do: set STOP_NOW or RESET_NOW (and CONFIRM) to use this cell.")

## 5. Live view

Read-only. Every `REFRESH_SECONDS` for `WATCH_MINUTES`, it shows the last `LOOKBACK_MINUTES` per target: CPU, the five largest wait categories, and the most blocked processes. Telemetry arrives 3 to 15 minutes after the activity, so a new run shows up after a few refreshes. For a picture that refreshes on its own, open Database Hub **Performance** or the Real-Time Dashboard (section 6).

In [ ]:
# Live view: read-only telemetry queries, refreshed until WATCH_MINUTES have passed.
refresh = min(max(int(REFRESH_SECONDS), 30), 300)
deadline = time.monotonic() + max(int(WATCH_MINUTES), 0) * 60
while True:
    frames, message = live_snapshot(LOOKBACK_MINUTES)
    print("=== %s UTC, the last %d minutes" % (dt.datetime.now(dt.timezone.utc).strftime("%H:%M"), int(LOOKBACK_MINUTES)))
    if frames is None:
        print(message)
        break
    show_snapshot(frames)
    if time.monotonic() + refresh > deadline:
        break
    time.sleep(refresh)

## 6. Links

- **Database Hub:** Performance shows the same telemetry with its own charts: https://powerbi.com/workloads/fdh/databaseHub
- **Real-Time Dashboard:** `RTD_DatabaseHub_LiveView` in the demo workspace refreshes on its own.
- **The runbook's job output:** the Automation account `aa-<prefix>-control` in the Azure portal, **Jobs**.
- **Without this notebook:** `./scripts/demo-control.ps1` sets an activity profile from your machine, and `./scripts/stop-demo.ps1` stops the demo.